# Proteina checkpoint browser

Pick a run + step from the shared registry, generate a protein, view it inline.

Pipeline mirrors `evaluation/proteina/generation/scripts/evaluate.py` but for a single sample.
Requires CUDA and the proteina .venv (`source .venv/bin/activate`).

## 1. Setup

In [ ]:
import os

os.environ.setdefault("DATA_PATH", "/rds/user/sr2173/hpc-work/proteina/data")

import sys
from pathlib import Path

REPO = Path("/home/sr2173/git/molecular-repa")
sys.path.insert(0, str(REPO / "src/proteina"))
import proteinfoundation.repa.pyg_compat  # noqa: F401,E402  -- patches torch_scatter sys.modules

sys.path.insert(0, str(REPO / "evaluation/proteina"))


# Repo-local fixes that the evaluate.sh wrapper normally sets up.
# torch.load weights_only=False shim for old proteina checkpoints.
import torch  # noqa: E402

_orig_load = torch.load


def _load(*a, **kw):
    kw["weights_only"] = False
    return _orig_load(*a, **kw)


torch.load = _load


from lib.checkpoints import (  # noqa: E402
    RUN_SCHEDULES,
    STORE_ROOT,
    find_checkpoint_path,
    resolve_step,
    load_checkpoint_by_path,
)

print("cuda:", torch.cuda.is_available(), "| store_root:", STORE_ROOT)

In [ ]:
import torch
import socket

print(socket.gethostname(), torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 2. List available checkpoints

Walks the actual `checkpoints/` directory for each run in the registry and surfaces every EMA step that exists on disk (so you're not limited to the log-spaced subset the eval sweep uses).

In [ ]:
import re

STEP_RE = re.compile(r"step=(\d+)-EMA\.ckpt$")


def list_checkpoints(run_name: str):
    run_dir, is_repa, layer, _ = RUN_SCHEDULES[run_name]
    ckpt_dir = STORE_ROOT / run_dir / "checkpoints"
    if not ckpt_dir.exists():
        return []
    steps = []
    for entry in os.listdir(ckpt_dir):
        m = STEP_RE.search(entry)
        if m:
            steps.append(int(m.group(1)))
    if (ckpt_dir / "last-EMA.ckpt").exists():
        steps.append(None)  # last-EMA fallback
    return sorted(steps, key=lambda s: (s is None, s or 0))


print(f'{"run":<24}  {"is_repa":<7}  {"layer":<5}  available_steps')
print("-" * 90)
for name, (run_dir, is_repa, layer, _) in RUN_SCHEDULES.items():
    steps = list_checkpoints(name)
    if not steps:
        summary = "(no checkpoints on disk)"
    else:
        non_none = [s for s in steps if s is not None]
        last = ", last-EMA" if any(s is None for s in steps) else ""
        if len(non_none) > 6:
            head = ", ".join(f"{s//1000}K" for s in non_none[:3])
            tail = ", ".join(f"{s//1000}K" for s in non_none[-3:])
            summary = f"{head}, ..., {tail} ({len(non_none)} steps){last}"
        else:
            summary = ", ".join(f"{s//1000}K" for s in non_none) + last
    print(f"{name:<24}  {str(is_repa):<7}  {layer:<5}  {summary}")

## 3. Choose checkpoint + sampling settings

Edit the cell below. `STEP=None` falls back to `last-EMA.ckpt`.

In [ ]:
# RUN_NAME = "baseline_128"  # any key from the table above
# STEP = 800000  # int, or None for last-EMA
RUN_NAME = "repa_l4_afdb_256_ep20"
STEP = None
NRES = 256  # protein length (residues)
NSAMPLES = 1  # how many to generate at once
SEED = 5

# Sampling hyperparams (defaults match inference_base.yaml)
DT = 0.0025
SELF_COND = True
SAMPLING_MODE = "sc"  # 'sc' (SDE) or 'vf' (ODE)
SC_SCALE_NOISE = 0.45
SCHEDULE_MODE = "log"
SCHEDULE_P = 2.0

run_dir, IS_REPA, _, _ = RUN_SCHEDULES[RUN_NAME]
ckpt_path = find_checkpoint_path(run_dir, STEP)
assert (
    ckpt_path is not None and ckpt_path.exists()
), f"no ckpt for {RUN_NAME} step={STEP}"
actual_step = resolve_step(ckpt_path, STEP)
print(f"will load: {ckpt_path}\n         step={actual_step}, is_repa={IS_REPA}")

## 4. Load model

In [ ]:
from omegaconf import OmegaConf

inf_cfg = OmegaConf.create(
    {
        "self_cond": SELF_COND,
        "fold_cond": False,
        "cath_code_level": "T",
        "guidance_weight": 1.0,
        "autoguidance_ratio": 0.0,
        "autoguidance_ckpt_path": None,
        "lora": {
            "use": False,
            "lora_alpha": 32.0,
            "lora_dropout": 0.0,
            "r": 16,
            "train_bias": "none",
        },
        "sampling_caflow": {
            "sampling_mode": SAMPLING_MODE,
            "sc_scale_noise": SC_SCALE_NOISE,
            "sc_scale_score": 1.0,
            "gt_mode": "1/t",
            "gt_p": 1.0,
            "gt_clamp_val": None,
        },
        "schedule": {"schedule_mode": SCHEDULE_MODE, "schedule_p": SCHEDULE_P},
    }
)

from lib.torch_load_patch import apply as _apply_torch_load_patch  # noqa: E402

_apply_torch_load_patch(strip_repa=not IS_REPA)

model = load_checkpoint_by_path(str(ckpt_path), is_repa=IS_REPA, device="cuda")
model.configure_inference(inf_cfg, nn_ag=None)
model.eval()
print("model loaded:", type(model).__name__)

## 5. Generate

First call at a new (batch_size, nres) compiles for ~60-90s if you enable `torch.compile`; it's left off here so a single sample comes back fast.

In [ ]:
import lightning as L
import time

L.seed_everything(SEED)
sc = inf_cfg["sampling_caflow"]
t0 = time.time()
with torch.no_grad():
    x = model.generate(
        nsamples=NSAMPLES,
        n=NRES,
        dt=DT,
        self_cond=SELF_COND,
        cath_code=None,
        guidance_weight=1.0,
        autoguidance_ratio=0.0,
        dtype=torch.float32,
        schedule_mode=SCHEDULE_MODE,
        schedule_p=SCHEDULE_P,
        sampling_mode=sc["sampling_mode"],
        sc_scale_noise=sc["sc_scale_noise"],
        sc_scale_score=sc["sc_scale_score"],
        gt_mode=sc["gt_mode"],
        gt_p=sc["gt_p"],
        gt_clamp_val=sc["gt_clamp_val"],
    )
    coors_atom37 = model.samples_to_atom37(x).cpu()
print(f"generated {coors_atom37.shape} in {time.time()-t0:.1f}s")

## 6. Save PDB(s)

In [ ]:
from proteinfoundation.utils.ff_utils.pdb_utils import write_prot_to_pdb

out_dir = Path("outputs") / f"{RUN_NAME}_step{actual_step}_n{NRES}_seed{SEED}"
out_dir.mkdir(parents=True, exist_ok=True)
pdb_paths = []
for i in range(coors_atom37.shape[0]):
    p = out_dir / f"sample_{i:03d}.pdb"
    write_prot_to_pdb(coors_atom37[i].numpy(), str(p), overwrite=True, no_indexing=True)
    pdb_paths.append(p)
for p in pdb_paths:
    print(p)

## 7. View

Inline cartoon with py3Dmol. Pick a sample index from the list above (`PDB_IDX`).

In [ ]:
import py3Dmol

PDB_IDX = 0
pdb_str = Path(pdb_paths[PDB_IDX]).read_text()

view = py3Dmol.view(width=600, height=500)
view.addModel(pdb_str, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()

from IPython.display import HTML  # noqa: E402

HTML(view._make_html())

In [ ]:
html_path = out_dir / f"sample_{PDB_IDX:03d}.html"
view.write_html(str(html_path))
print("open:", html_path)